In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"H:\HTOC\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: H:\\HTOC\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_DAYS = 90  # days of lastObserved activity to include
INDICATOR_TYPE_NAMES = [
    "Address", "EmailAddress", "Host", "URL"
]
OWNER_NAMES = [
    'Google Threat Intelligence'
]
RESULT_PAGE_SIZE = 5_000  # benchmarked stable: ~310 rows/sec with the same fields
MAX_TOTAL_RESULTS = 5_000  # hard cap on records returned by this pull

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=QUERY_LOOKBACK_DAYS)).date()
start = f"{start_date}T00:00:00Z"

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) AND '
    f'DateAdded >= "{start}" AND '
    'observationCount = 0 AND '
    'rating >= 3 AND '
    'confidence >= 50 AND '
    'source IS NOT NULL'
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while result_start < MAX_TOTAL_RESULTS:
    current_limit = min(result_limit, MAX_TOTAL_RESULTS - result_start)
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,falsePositives,threatAssess'
            f'&resultStart={result_start}&resultLimit={current_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += len(data_items)

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
    # Filter to keep only records where ownerName is 'HTOC Org'
    #observed_src = observed_src[observed_src['ownerName'] == 'HTOC Org'].copy()
    # Keep rows where top-level rating >= 3 OR coalesced threatAssessRating >= 3, and
    # (coalesced TA confidence >= 50 OR top-level confidence >= 50).
    # Coalesce flat vs nested threatAssess columns; keep top-level rating/confidence separate for OR.
    _rating_cols = ("threatAssessRating", "threatAssess.threatAssessRating", "rating")
    _confidence_cols = ("threatAssessConfidence", "threatAssess.threatAssessConfidence")

    def _first_non_null_numeric(df, ordered_cols):
        present = [c for c in ordered_cols if c in df.columns]
        if not present:
            return None
        out = pd.to_numeric(df[present[0]], errors="coerce")
        for c in present[1:]:
            s = pd.to_numeric(df[c], errors="coerce")
            out = out.mask(out.isna(), s)
        return out

    _tar = _first_non_null_numeric(observed_src, _rating_cols)
    _tc = _first_non_null_numeric(observed_src, _confidence_cols)
    if _tar is None or _tc is None:
        raise KeyError(
            f"Could not resolve Threat Assess columns. Tried rating={_rating_cols}, "
            f"confidence={_confidence_cols}. Columns: {list(observed_src.columns)}"
        )
    if "rating" in observed_src.columns:
        _r = pd.to_numeric(observed_src["rating"], errors="coerce")
    else:
        _r = pd.Series(float("nan"), index=observed_src.index, dtype=float)

    if "confidence" in observed_src.columns:
        _c = pd.to_numeric(observed_src["confidence"], errors="coerce")
    else:
        _c = pd.Series(float("nan"), index=observed_src.index, dtype=float)

    _pre_ta = len(observed_src)
    # Use >= 50 so a boundary value of 50.0 is included (strict > 50 dropped those rows).
    _pass_rating_band = (_tar >= 3) | (_r >= 3)
    _pass_confidence_band = (_tc >= 50) | (_c >= 50)
    observed_src = observed_src[_pass_rating_band & _pass_confidence_band].copy()
    display(
        f"Threat assess filter ((rating>=3 OR threatAssessRating>=3), confidence>=50) coalescing {_rating_cols} / {_confidence_cols}: "
        f"{_pre_ta} -> {len(observed_src)} rows."
    )
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)

## GTI 90-day promotion estimate (2026-10-05)

- Exact initial TQL population: **459,193**
- Estimated to pass the revised rules: **45,646 (9.9%)**
- Approximate 95% sampling interval: **37,328–53,965**
- Method: exact API count plus a date-sorted, type-stratified systematic sample of 1,000 enriched indicators (250 Address, 250 Host, and 500 URL), weighted by the full 90-day type population.

The revised analysis accepts every value already typed as URL by ThreatConnect, requires a qualified Malware/Campaign/Intrusion Set/Report association, requires at least two distinct non-empty source tags, and requires only a non-empty Description. Tag names are counted case-insensitively without renaming or deriving tags. Rating and confidence thresholds are unchanged.


In [2]:
import json
import math
import re
import sys
import urllib.parse
from collections import Counter
from datetime import datetime, timedelta, timezone

sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
sys.path.append(r"H:\HTOC\scripts\Data Movement\ThrearConnect-api-pull")

from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from utils.config_loader import load_config

CONFIG_PATH = r"H:\HTOC\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json"
TYPES = ("Address", "EmailAddress", "Host", "URL")
TARGET_SAMPLE_BY_TYPE = {"Address": 250, "EmailAddress": 0, "Host": 250, "URL": 500}
PAGE_SIZE = 25
FIELDS = "attributes,tags,associatedGroups,falsePositives,threatAssess"


def request_json(tc, uri):
    request = RequestObject()
    request.set_http_method("GET")
    request.set_request_uri(uri)
    response = tc.api_request(request)
    if not response.headers.get("content-type", "").startswith("application/json"):
        raise RuntimeError(f"Non-JSON response: {response.status_code}")
    payload = response.json()
    if payload.get("status") == "Error":
        raise RuntimeError(payload.get("message", "ThreatConnect API error"))
    return payload


def uri_for(tql, result_start=0, result_limit=1, count=False):
    parts = [
        f"tql={urllib.parse.quote(tql)}",
        f"fields={urllib.parse.quote(FIELDS)}",
        f"resultStart={result_start}",
        f"resultLimit={result_limit}",
    ]
    if count:
        parts.append("count=true")
    parts.append(f"sorting={urllib.parse.quote('dateAdded ASC')}")
    return "/v3/indicators?" + "&".join(parts)


def nested_data(record, key):
    value = record.get(key) or {}
    return value.get("data", []) if isinstance(value, dict) else []


def description(record):
    for attribute in nested_data(record, "attributes"):
        if attribute.get("type") == "Description":
            return str(attribute.get("value") or "")
    return ""


SYNTAX = {
    "Address": re.compile(r"^(?:(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)|[0-9A-Fa-f:]*:[0-9A-Fa-f:]+)$"),
    "EmailAddress": re.compile(r"^[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?(?:\.[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?)+$"),
    "Host": re.compile(r"^(?=.{1,253}$)(?:[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?\.)+[A-Za-z]{2,63}$"),
    "URL": re.compile(r"^.+$"),  # Accept every URL style already stored by ThreatConnect.
}
BOILERPLATE = re.compile(
    r"did not match our detection criteria|there is currently no evidence of malicious activity|"
    r"^\s*(?:n/?a|none|unknown|placeholder|test)\s*$|"
    r"^\s*this indicator is (?:suspicious|malicious)\s*\([^)]*\)\.?"
    r"(?:\s*(?:its verdict recently changed|gti's ml scoring model identified).*)?\s*$",
    re.I | re.S,
)
WEAK_TAG = re.compile(
    r"^(?:ip|address|host|url|email(?:address)?|ns-port|external-resources|"
    r"google threat intelligence|gti|indicator|unknown|uncategorized|none|n/?a)$",
    re.I,
)
STRONG_TAG = re.compile(
    r"malware|phish|dga|fraud|bot(?:net)?|c2|command(?:-|\s+)?and(?:-|\s+)?control|"
    r"infect|spyware|ransom|trojan|stealer|loader|backdoor|\brat\b|campaign|threat|malicious",
    re.I,
)


def threat_assess(record):
    threat = record.get("threatAssess") or {}
    rating = record.get("threatAssessRating")
    confidence = record.get("threatAssessConfidence")
    if isinstance(threat, dict):
        rating = rating if rating is not None else threat.get("threatAssessRating")
        confidence = confidence if confidence is not None else threat.get("threatAssessConfidence")
    try:
        return float(rating) >= 3 and float(confidence) >= 50
    except (TypeError, ValueError):
        return False


def source_valid(record):
    source = str(record.get("source") or "")
    indicator_type = record.get("type")
    prefixes = {
        "Address": "https://www.virustotal.com/gui/ip-address/",
        "Host": "https://www.virustotal.com/gui/domain/",
        "URL": "https://www.virustotal.com/gui/url/",
    }
    return indicator_type in prefixes and source.startswith(prefixes[indicator_type])


def context_valid(record):
    groups = nested_data(record, "associatedGroups")
    qualified_group = any(
        group.get("id") not in (None, "")
        and str(group.get("name") or "").strip()
        and group.get("type") in {"Malware", "Campaign", "Intrusion Set", "Report"}
        for group in groups
    )
    return qualified_group


def first_failure(record):
    if not threat_assess(record):
        return "threat_assessment"
    value = str(record.get("summary") or "")
    indicator_type = record.get("type")
    if indicator_type not in SYNTAX or not SYNTAX[indicator_type].fullmatch(value):
        return "syntax"
    text = description(record)
    if not text.strip():
        return "description"
    tags = (record.get("tags") or {}).get("data", [])
    distinct_tags = {
        str(tag.get("name") or "").strip().casefold()
        for tag in tags
        if str(tag.get("name") or "").strip()
    }
    if len(distinct_tags) < 2:
        return "minimum_tags"
    if not context_valid(record):
        return "analytic_context"
    if not source_valid(record):
        return "source"
    required = (
        record.get("id"), indicator_type, value, record.get("dateAdded"),
        record.get("ownerName"), record.get("source"), record.get("rating"),
        record.get("confidence"), text,
    )
    if any(item in (None, "") for item in required):
        return "base_or_provenance"
    return "pass"


def sample_offsets(population, indicator_type):
    target = TARGET_SAMPLE_BY_TYPE[indicator_type]
    if target == 0 or population == 0:
        return []
    if population <= PAGE_SIZE:
        return [0]
    maximum = population - PAGE_SIZE
    pages = max(1, target // PAGE_SIZE)
    if pages == 1:
        return [0]
    return sorted({round(maximum * i / (pages - 1)) for i in range(pages)})


def main():
    secret, access_id, base_url, default_org = load_config(CONFIG_PATH)
    tc = ThreatConnect(access_id, secret, default_org, base_url)
    start = (datetime.now(timezone.utc) - timedelta(days=90)).date().isoformat() + "T00:00:00Z"
    base_tql = (
        'ownerName = "Google Threat Intelligence" AND '
        'typeName IN ("Address", "EmailAddress", "Host", "URL") AND '
        f'DateAdded >= "{start}" AND '
        'falsePositiveCount = 0 AND observationCount = 0 AND '
        'rating >= 3 AND confidence >= 50 AND source IS NOT NULL'
    )

    initial = request_json(tc, uri_for(base_tql, count=True))
    total_count = int(initial["count"])
    type_counts = {}
    samples = {}
    results = {}

    for indicator_type in TYPES:
        type_tql = base_tql + f' AND typeName = "{indicator_type}"'
        count_payload = request_json(tc, uri_for(type_tql, count=True))
        population = int(count_payload["count"])
        type_counts[indicator_type] = population
        records = {}
        for offset in sample_offsets(population, indicator_type):
            page = request_json(tc, uri_for(type_tql, result_start=offset, result_limit=PAGE_SIZE))
            for record in page.get("data", []):
                records[record["id"]] = record
        samples[indicator_type] = list(records.values())
        results[indicator_type] = Counter(first_failure(record) for record in samples[indicator_type])

    estimate = 0.0
    variance = 0.0
    for indicator_type in TYPES:
        population = type_counts[indicator_type]
        sample_size = len(samples[indicator_type])
        passed = results[indicator_type]["pass"]
        rate = passed / sample_size if sample_size else 0.0
        estimate += population * rate
        if sample_size > 1:
            sample_variance = rate * (1 - rate) * sample_size / (sample_size - 1)
            finite_correction = max(0.0, 1 - sample_size / population) if population else 0.0
            variance += population * population * finite_correction * sample_variance / sample_size

    standard_error = math.sqrt(variance)
    report = {
        "run_utc": datetime.now(timezone.utc).isoformat(),
        "lookback_start_utc": start,
        "initial_pull_count": total_count,
        "type_counts": type_counts,
        "sample_size": sum(len(value) for value in samples.values()),
        "sample_size_by_type": {key: len(value) for key, value in samples.items()},
        "sample_results_by_type": {key: dict(value) for key, value in results.items()},
        "estimated_move_count": round(estimate),
        "estimated_move_rate": estimate / total_count if total_count else 0.0,
        "approx_95_percent_interval": [
            max(0, round(estimate - 1.96 * standard_error)),
            min(total_count, round(estimate + 1.96 * standard_error)),
        ],
        "method": "Date-sorted type-stratified systematic sample of 1,000 records; estimates weighted by the 90-day type population.",
        "tql": base_tql,
    }
    print(json.dumps(report, indent=2, sort_keys=True))


if __name__ == "__main__":
    main()


{
  "approx_95_percent_interval": [
    37328,
    53965
  ],
  "estimated_move_count": 45646,
  "estimated_move_rate": 0.09940560069513255,
  "initial_pull_count": 459193,
  "lookback_start_utc": "2026-07-07T00:00:00Z",
  "method": "Date-sorted type-stratified systematic sample of 1,000 records; estimates weighted by the 90-day type population.",
  "run_utc": "2026-10-05T18:40:26.412604+00:00",
  "sample_results_by_type": {
    "Address": {
      "pass": 15
    },
    "EmailAddress": {},
    "Host": {
      "pass": 85
    },
    "URL": {
      "pass": 22
    }
  },
  "sample_size": 1000,
  "sample_size_by_type": {
    "Address": 250,
    "EmailAddress": 0,
    "Host": 250,
    "URL": 500
  },
  "type_counts": {
    "Address": 5351,
    "EmailAddress": 0,
    "Host": 85663,
    "URL": 368179
  }
}
